<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 02 - Técnicas de Pré-Processamento de Texto</h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2> Técnicas para Pré-Processamento </h2>

<p>Vamos avaliar as técnicas mais comuns para prepararmos o texto para usar com algoritmos de aprendizado de máquina logo mais.</p>
<p>Como estudo de caso, vamos usar o texto de <i>Hamlet</i>, encontrado no corpus <i>Gutenberg</i> do pacote <b>NLTK</b></p>

<b>1. Baixando o corpus Gutenberg</b>

In [ ]:
import nltk

nltk.download("gutenberg")

<b>2. Exibindo o texto "Hamlet"</b>

In [ ]:
hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')
print(hamlet_raw[:1000])

<b>3. Segmentação de sentenças e tokenização de palavras</b>

Consiste em dividir o texto em sentenças. Porém, não é uma tarefa tão trivial quanto usar uma função split(".") no Python, por exemplo. Isso porque o uso do ponto (.) nem sempre é empregado como ponto final em uma sentença. Por exemplo:

"Dr. Rodolfo S. Silva, um renomado cardiologista, adquiriu um novo consultório na Av. Edson A. Nascimento pela quantia de R$ 4.5 milhões, que atenderá pacientes do S.U.S. e do plano Saúde Mais Inc."

Uma das técnicas mais conhecidas para resolver esse problema é o algoritmo Punkt. Trata-se de um modelo que consiste em identificar abreviações, reticências, iniciais e números ordinais para, aí então, identificar os pontos finais em sentenças.

![Arquitetura Punkt](https://www.researchgate.net/profile/Jan_Strunk/publication/220355311/figure/fig1/AS:277323623485449@1443130512700/Architecture-of-the-Punkt-System.png)

In [ ]:
from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(hamlet_raw)

print(sentences[:10])


In [ ]:
from nltk.tokenize import word_tokenize

words = word_tokenize(sentences[0])

print(words)

<b>4. Removendo stopwords e pontuação</b>

In [ ]:
from nltk.corpus import stopwords

stopwords_list = stopwords.words('english')

print(stopwords_list)

In [ ]:
non_stopwords = [w for w in words if not w.lower() in stopwords_list]
print(non_stopwords)

In [ ]:
import string
punctuation = string.punctuation
print(punctuation)

In [ ]:
 

print(non_punctuation)

<b>5. Part of Speech (POS) Tags </b>

In [ ]:
from nltk import pos_tag

pos_tags = pos_tag(words)#com o texto original

print(pos_tags)

As tags indicam a classificação sintática de cada palavra no texto. Ver <a href="https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html" target="blank">https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html</a> para uma lista completa

<b>6. Stemming e Lemmatization</b>

stemming traz a raiz (indi=i, fazendo=fase)
Lemmatization consulta na base do wordnet e procura a raiz, e traz no infinitivo

Stemming permite obter a "raiz" da palavra, removendo sufixos, por exemplo.

In [ ]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer('english')

sample_sentence = "He has already gone"
sample_words = word_tokenize(sample_sentence)

stems = [stemmer.stem(w) for w in sample_words]

print(stems)

Já lemmatization vai além de somente remover sufixos, obtendo a raiz linguística da palavra. Vamos usar as tags POS obtidas anteriormente para otimizar o lemmatizer.

In [ ]:
nltk.download('wordnet')

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer() #precisa do POS

pos_tags = nltk.pos_tag(sample_words)

lemmas = []
for w in pos_tags:
    if w[1].startswith('J'):
        pos_tag = wordnet.ADJ
    elif w[1].startswith('V'):
        pos_tag = wordnet.VERB
    elif w[1].startswith('N'):
        pos_tag = wordnet.NOUN
    elif w[1].startswith('R'):
        pos_tag = wordnet.ADV
    else:
        pos_tag = wordnet.NOUN
        
    lemmas.append(lemmatizer.lemmatize(w[0], pos_tag))
    
print(lemmas)

<b>7. N-gramas</b>

Além da técnica de <i>Bag-of-Words</i>, outra opção é utilizar n-gramas (onde "n" pode variar)

In [ ]:
from nltk import word_tokenize

frase = 'o cachorro correu atrás do gato'


ngrams = ["%s %s %s" % (nltk.word_tokenize(frase)[i], \
                      nltk.word_tokenize(frase)[i+1], \
                      nltk.word_tokenize(frase)[i+2]) \
          for i in range(len(nltk.word_tokenize(frase))-2)]#para cada palavra dentor das palavras a té a antepenúltima

print(ngrams)


In [ ]:
non_punctuation = [w for w in words if not w.lower() in punctuation]

n_grams_3 = ["%s %s %s"%(non_punctuation[i], non_punctuation[i+1], non_punctuation[i+2]) for i in range(0, len(non_punctuation)-2)]

print(n_grams_3)

Também podemos usar a classe <b>CountVectorizer</b>, do scikit-learn:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vect = CountVectorizer(ngram_range=(3,3))#de trigramas a trigramas

import numpy as np

#arr = np.array(sentences[:10])#as 10 primeiras
arr = np.array(sentences[:10])#todas

print(arr)

n_gram_counts = count_vect.fit_transform(arr)#extrai os n-gramas e conta a frequência

print(n_gram_counts.toarray()[:10])#lista de contagens

print(count_vect.vocabulary_)#vocabulário
#cada linha é uma sentença
#cada coluna é se há trigrama na sentença, se houver 2 "1" na mesma coluna este trigrama está em duas sentenças
#a posição do 1 é o índice do trigrama que está impresso na lista de contagens
#se houver o mesmo trigrama na mesma sentença, na linha, ele soma

Agora, vamos contar os n-grams (no nosso caso, trigramas) de todas as sentenças do texto:

In [ ]:
arr = np.array(sentences)

n_gram_counts = count_vect.fit_transform(arr)

print(n_gram_counts.toarray()[:20])

print([k for k in count_vect.vocabulary_.keys()][:20])

<p><b>Exercício 2:</b>Exiba 10 lemmas mais frequentes do corpus Reuters, ignorando pontuações e stopwords.</p>



In [100]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from operator import itemgetter
import pandas as pd
import operator 
import itertools

In [101]:
words = [w for w in (sent_tokenize(nltk.corpus.reuters.raw()))]

In [ ]:
sentences = sent_tokenize(reuters)
print(sentences)[:100]

In [102]:
#words = [w for w in sentences]

print(words)[:100]

["ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RIFT\n  Mounting trade friction between the\n  U.S. And Japan has raised fears among many of Asia's exporting\n  nations that the row could inflict far-reaching economic\n  damage, businessmen and officials said.", 'They told Reuter correspondents in Asian capitals a U.S.\n  Move against Japan might boost protectionist sentiment in the\n  U.S. And lead to curbs on American imports of their products.', "But some exporters said that while the conflict would hurt\n  them in the long-run, in the short-term Tokyo's loss might be\n  their gain.", "The U.S. Has said it will impose 300 mln dlrs of tariffs on\n  imports of Japanese electronics goods on April 17, in\n  retaliation for Japan's alleged failure to stick to a pact not\n  to sell semiconductors on world markets at below cost.", 'Unofficial Japanese estimates put the impact of the tariffs\n  at 10 billion dlrs and spokesmen for major electronics firms\n  said they would virtually halt expo

TypeError: 'NoneType' object is not subscriptable

In [ ]:
#tokenized = [word_tokenize(i) for i in words]
#tokenized[:100]
tokenized = []
for w in words:
    tokenized.extend(word_tokenize(w))
tokenized[:100]
    

In [ ]:
#lists = []
#for l in tokenized:
 #   lists.append(str.join(',',l))
#lists[:100]

In [ ]:
pos_tags = nltk.pos_tag(tokenized)
pos_tags[:100]

In [ ]:
#nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmas = []
for w in pos_tags:       
    
    if w[1].startswith('J'):
        pos_tag = wordnet.ADJ
    elif w[1].startswith('V'):
        pos_tag = wordnet.VERB
    elif w[1].startswith('N'):
        pos_tag = wordnet.NOUN
    elif w[1].startswith('R'):
        pos_tag = wordnet.ADV
    else:
        pos_tag = wordnet.NOUN

    lemmas.append(lemmatizer.lemmatize(w[0], pos_tag))
    
print(lemmas)[:100]

In [ ]:
non_stopwords = [w for w in lemmas if not w[0].lower() in stopwords_list]
    
non_punctuation = [w for w in non_stopwords if not w[0] in string.punctuation]

In [68]:
non_punctuation

['EXPORTERS',
 'FEAR',
 'FROM',
 'U.S.-JAPAN',
 'RIFT',
 'friction',
 'between',
 'U.S.',
 'Japan',
 'have',
 'raise',
 'fear',
 'export',
 'nation',
 'row',
 'could',
 'far-reaching',
 'economic',
 'businessmen',
 'Reuter',
 'correspondent',
 'capital',
 'U.S.',
 'Japan',
 'boost',
 'protectionist',
 'U.S.',
 'lead',
 'curb',
 'product',
 'But',
 'exporter',
 'while',
 'conflict',
 'would',
 'hurt',
 'long-run',
 'loss',
 'be',
 'gain',
 'U.S.',
 'Has',
 'will',
 '300',
 'Japanese',
 'electronics',
 'good',
 '17',
 'retaliation',
 'for',
 'Japan',
 'failure',
 'pact',
 'not',
 'world',
 'below',
 'cost',
 'Unofficial',
 'Japanese',
 'estimate',
 'put',
 '10',
 'billion',
 'for',
 'electronics',
 'firm',
 'would',
 'virtually',
 'halt',
 'export',
 'product',
 'hit',
 'by',
 'new',
 'We',
 'would',
 "n't",
 'be',
 'business',
 'for',
 'lead',
 'Japanese',
 'electronics',
 'firm',
 'Electric',
 'Co',
 'Ltd',
 'lt',
 'remain',
 'place',
 'for',
 'length',
 'beyond',
 'few',
 'will',
 'co

In [99]:

print(sorted(non_punctuation.items(),key=operator.itemgetter(1)[-10:]))

AttributeError: 'str' object has no attribute 'items'

In [91]:
print(sorted(non_punctuation[:10]))

['EXPORTERS', 'FEAR', 'FROM', 'Japan', 'RIFT', 'U.S.', 'U.S.-JAPAN', 'between', 'friction', 'have']


In [108]:
print(non_punctuation.sort(key=max))

None
